In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def load_dataset(file_path):
    df = pd.read_csv(file_path)
    df.rename(columns={"Time [ms]": "time", "Moving1.Torque [NewtonMeter]": "torque"}, inplace=True)
    df['time'] = df['time'] / 1000  
    df['fault_label'] = df['operating condition'].apply(lambda x: 1 if x == "fault" else 0)
    return df

file_path = "C:\\Users\\ravir\\Downloads\\updated_srm_dataset.csv"
df = load_dataset(file_path)

features = ['torque', 'Current(PhaseA) [A]', 'Current(PhaseB) [A]', 'Current(PhaseC) [A]', 'Current(PhaseD) [A]']
target = 'fault_label'

X = df[features]
y = df[target]

# Store average currents during normal condition
normal_df = df[df['fault_label'] == 0]
avg_currents = {
    'A': normal_df['Current(PhaseA) [A]'].mean(),
    'B': normal_df['Current(PhaseB) [A]'].mean(),
    'C': normal_df['Current(PhaseC) [A]'].mean(),
    'D': normal_df['Current(PhaseD) [A]'].mean()
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

def get_fault_severity(deviation):
    if deviation < 4:
        return "Low"
    elif deviation < 8:
        return "Moderate"
    elif deviation < 12:
        return "High"
    elif deviation < 16:
        return "Severely High"
    else:
        return "Extremely Severe"

def predict_fault():
    try:
        torque = float(input("Enter Moving Torque value: "))
        current_a = float(input("Enter Phase A Current: "))
        current_b = float(input("Enter Phase B Current: "))
        current_c = float(input("Enter Phase C Current: "))
        current_d = float(input("Enter Phase D Current: "))
        
        input_data = np.array([[torque, current_a, current_b, current_c, current_d]])
        prediction = model.predict(input_data)
        
        if prediction[0] == 1:
            print("\n🔴 Fault Detected!")

            deviations = {
                'A': abs(current_a - avg_currents['A']),
                'B': abs(current_b - avg_currents['B']),
                'C': abs(current_c - avg_currents['C']),
                'D': abs(current_d - avg_currents['D']),
            }

            # Identify phase with max deviation
            faulty_phase = max(deviations, key=deviations.get)
            deviation_value = deviations[faulty_phase]
            severity = get_fault_severity(deviation_value)

            print(f"⚠ Fault likely in Phase {faulty_phase}")
            print(f"🟠 Fault Severity: {severity}")
        else:
            print("\n✅ SRM Condition: NORMAL")
    
    except Exception as e:
        print(f"❌ Error in prediction: {e}")

predict_fault()
